## Apply bias mitigation techniques if necessary by implementing reweighting, adversarial debiasing, or preprocessing corrections, testing multiple bias-removal strategies and quantifying their effects on both fairness and performance.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:

df = pd.read_csv("heart.csv")
df = df.drop_duplicates().dropna().reset_index(drop=True)

y = df["target"].astype(int)
raw_X = df.drop(columns=["target"])

if "sex" not in raw_X.columns:
    raise ValueError("The dataset must contain a 'sex' column.")

X = pd.get_dummies(raw_X, drop_first=True)

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

train_idx, valid_idx = train_test_split(
    train_idx,
    test_size=0.25,
    random_state=42,
    stratify=y.iloc[train_idx]
)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_valid = X.iloc[valid_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)

y_train = y.iloc[train_idx].reset_index(drop=True)
y_valid = y.iloc[valid_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

sex_train = raw_X.iloc[train_idx]["sex"].reset_index(drop=True)
sex_valid = raw_X.iloc[valid_idx]["sex"].reset_index(drop=True)
sex_test = raw_X.iloc[test_idx]["sex"].reset_index(drop=True)

X_train_no_sex = X_train.drop(columns=["sex"], errors="ignore")
X_valid_no_sex = X_valid.drop(columns=["sex"], errors="ignore")
X_test_no_sex = X_test.drop(columns=["sex"], errors="ignore")

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Testing rows:", len(X_test))


Training rows: 180
Validation rows: 61
Testing rows: 61


In [4]:
def fairness_metrics(y_true, y_pred, groups):
    data = pd.DataFrame({
        "actual": np.asarray(y_true),
        "predicted": np.asarray(y_pred),
        "group": np.asarray(groups)
    })

    group_rates = data.groupby("group")["predicted"].mean()
    tpr_values = []
    fpr_values = []

    for group in data["group"].dropna().unique():
        subset = data[data["group"] == group]
        actual = subset["actual"]
        predicted = subset["predicted"]

        tp = ((actual == 1) & (predicted == 1)).sum()
        fn = ((actual == 1) & (predicted == 0)).sum()
        fp = ((actual == 0) & (predicted == 1)).sum()
        tn = ((actual == 0) & (predicted == 0)).sum()

        tpr_values.append(tp / (tp + fn) if tp + fn else np.nan)
        fpr_values.append(fp / (fp + tn) if fp + tn else np.nan)

    di = (
        group_rates.min() / group_rates.max()
        if len(group_rates) >= 2 and group_rates.max() > 0
        else np.nan
    )

    tpr_gap = np.nanmax(tpr_values) - np.nanmin(tpr_values) if np.isfinite(tpr_values).any() else np.nan
    fpr_gap = np.nanmax(fpr_values) - np.nanmin(fpr_values) if np.isfinite(fpr_values).any() else np.nan

    return {
        "Disparate Impact": di,
        "TPR Difference": tpr_gap,
        "FPR Difference": fpr_gap,
        "Equalized Odds Difference": max(tpr_gap, fpr_gap)
        if pd.notna(tpr_gap) and pd.notna(fpr_gap) else np.nan
    }

In [5]:
baseline = RandomForestClassifier(random_state=42)
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, baseline_pred))
print(fairness_metrics(y_test, baseline_pred, sex_test))

Baseline Accuracy: 0.7377049180327869
{'Disparate Impact': 0.5738880918220947, 'TPR Difference': np.float64(0.2536764705882353), 'FPR Difference': np.float64(0.026666666666666672), 'Equalized Odds Difference': np.float64(0.2536764705882353)}


In [6]:
df = pd.read_csv("heart.csv")
df = df.drop_duplicates().dropna().reset_index(drop=True)

y = df["target"].astype(int)
raw_X = df.drop(columns=["target"])

if "sex" not in raw_X.columns:
    raise ValueError("The dataset must contain a 'sex' column.")

X = pd.get_dummies(raw_X, drop_first=True)

train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=y
)

train_idx, valid_idx = train_test_split(
    train_idx, test_size=0.25, random_state=42, stratify=y.iloc[train_idx]
)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_valid = X.iloc[valid_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)

y_train = y.iloc[train_idx].reset_index(drop=True)
y_valid = y.iloc[valid_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

sex_train = raw_X.iloc[train_idx]["sex"].reset_index(drop=True)
sex_valid = raw_X.iloc[valid_idx]["sex"].reset_index(drop=True)
sex_test = raw_X.iloc[test_idx]["sex"].reset_index(drop=True)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Testing rows:", len(X_test))

Training rows: 180
Validation rows: 61
Testing rows: 61


In [7]:
train_info = pd.DataFrame({
    "sex": sex_train,
    "target": y_train
})

cell_counts = train_info.groupby(["sex", "target"])["target"].transform("size")
group_counts = train_info.groupby("sex")["target"].transform("size")

sample_weights = group_counts / cell_counts
sample_weights = sample_weights / sample_weights.mean()

reweighted_model = RandomForestClassifier(random_state=42)
reweighted_model.fit(X_train, y_train, sample_weight=sample_weights)

reweighted_pred = reweighted_model.predict(X_test)

print("Reweighted Accuracy:", accuracy_score(y_test, reweighted_pred))
print(fairness_metrics(y_test, reweighted_pred, sex_test))

Reweighted Accuracy: 0.7704918032786885
{'Disparate Impact': 0.5738880918220947, 'TPR Difference': np.float64(0.19117647058823528), 'FPR Difference': np.float64(0.013333333333333308), 'Equalized Odds Difference': np.float64(0.19117647058823528)}


In [8]:
X_train_no_sex = X_train.drop(columns=["sex"], errors="ignore")
X_valid_no_sex = X_valid.drop(columns=["sex"], errors="ignore")
X_test_no_sex = X_test.drop(columns=["sex"], errors="ignore")

no_sex_model = RandomForestClassifier(random_state=42)
no_sex_model.fit(X_train_no_sex, y_train)

no_sex_pred = no_sex_model.predict(X_test_no_sex)

print("Without Sex Accuracy:", accuracy_score(y_test, no_sex_pred))
print(fairness_metrics(y_test, no_sex_pred, sex_test))

Without Sex Accuracy: 0.7540983606557377
{'Disparate Impact': 0.6097560975609756, 'TPR Difference': np.float64(0.13235294117647056), 'FPR Difference': np.float64(0.013333333333333308), 'Equalized Odds Difference': np.float64(0.13235294117647056)}


In [9]:
from sklearn.metrics import confusion_matrix

valid_prob = baseline.predict_proba(X_valid)[:, 1]

thresholds = {}

for group in sex_valid.unique():
    best_threshold = 0.5
    best_gap = float("inf")

    mask = sex_valid.to_numpy() == group

    for threshold in np.arange(0.1, 0.91, 0.05):
        pred = (valid_prob[mask] >= threshold).astype(int)
        actual = y_valid.to_numpy()[mask]

        tn, fp, fn, tp = confusion_matrix(
            actual, pred, labels=[0, 1]
        ).ravel()

        tpr = tp / (tp + fn) if tp + fn else np.nan
        fpr = fp / (fp + tn) if fp + tn else np.nan

        if np.isfinite(tpr) and np.isfinite(fpr):
            gap = abs(tpr - fpr)

            if gap < best_gap:
                best_gap = gap
                best_threshold = threshold

    thresholds[group] = best_threshold

test_prob = baseline.predict_proba(X_test)[:, 1]

adjusted_pred = np.array([
    int(prob >= thresholds[group])
    for prob, group in zip(test_prob, sex_test)
])

print("Group thresholds:", thresholds)
print("Threshold-adjusted Accuracy:", accuracy_score(y_test, adjusted_pred))
print(fairness_metrics(y_test, adjusted_pred, sex_test))

Group thresholds: {np.int64(1): np.float64(0.9000000000000002), np.int64(0): np.float64(0.1)}
Threshold-adjusted Accuracy: 0.7213114754098361
{'Disparate Impact': 0.04878048780487805, 'TPR Difference': np.float64(0.875), 'FPR Difference': np.float64(1.0), 'Equalized Odds Difference': np.float64(1.0)}


In [10]:
predictions_by_strategy = {
    "Baseline": baseline_pred,
    "Reweighting": reweighted_pred,
    "Remove Sex": no_sex_pred,
    "Threshold Adjustment": adjusted_pred
}

comparison = []

for name, pred in predictions_by_strategy.items():
    fairness = fairness_metrics(y_test, pred, sex_test)

    comparison.append({
        "Strategy": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0),
        **fairness
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.round(3).to_string(index=False))

            Strategy  Accuracy  Precision  Recall  F1 Score  Disparate Impact  TPR Difference  FPR Difference  Equalized Odds Difference
            Baseline     0.738      0.730   0.818     0.771             0.574           0.254           0.027                      0.254
         Reweighting     0.770      0.757   0.848     0.800             0.574           0.191           0.013                      0.191
          Remove Sex     0.754      0.750   0.818     0.783             0.610           0.132           0.013                      0.132
Threshold Adjustment     0.721      0.864   0.576     0.691             0.049           0.875           1.000                      1.000


In [11]:
comparison_df.to_csv("Bias_Mitigation_Comparison.csv", index=False)

with open("Bias_Mitigation_Report.txt", "w") as file:
    file.write("BIAS MITIGATION ANALYSIS\n\n")
    file.write(comparison_df.round(4).to_string(index=False))
    file.write("\n\nThresholds selected on validation data:\n")
    file.write(str(thresholds))
    file.write("\n\nInterpretation:\n")
    file.write("Compare fairness metrics and predictive performance together.\n")
    file.write("Disparate Impact closer to 1 indicates more similar positive prediction rates.\n")
    file.write("Equalized Odds Difference closer to 0 indicates smaller TPR and FPR gaps.\n")
    file.write("Metrics alone do not prove that a model is fair or clinically safe.\n")

print("Reports saved successfully.")

Reports saved successfully.
